#### **2.1 Mounting Google Drive**

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/bone-fracture-detection'
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### **2.2 Pulling from GitHub**

In [2]:
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
os.environ['GH_TOKEN'] = token

username = "baotle"
repo_name = "BoneFractureDetection_Computervision_Project"

!git remote set-url origin https://github.com/{username}/{repo_name}.git
!git -c credential.helper='!f() { echo "username=x-access-token"; echo "password=$GH_TOKEN"; }; f' pull origin main


From https://github.com/baotle/BoneFractureDetection_Computervision_Project
 * branch            main       -> FETCH_HEAD
Already up to date.


### **2.2.1 Check for leakes in git commits**


#### **2.3 Pushing to GitHub (to be reused)**

In [11]:
token = userdata.get('GITHUB_TOKEN')
os.environ['GH_TOKEN'] = token

username = "baotle"
repo_name = "BoneFractureDetection_Computervision_Project"

!git remote set-url origin https://github.com/{username}/{repo_name}.git
!git add .
!git commit -m "{commit_message}"
!git -c credential.helper='!f() { echo "username=x-access-token"; echo "password=$GH_TOKEN"; }; f' push -q

print("✅ Pushed.")

[main fb18e43] {commit_message}
 1 file changed, 1 insertion(+), 1 deletion(-)
✅ Pushed.


#### **2.4 Install torch and check for device =GPU**

In [6]:
import torch
print(f"GPU available : {torch.cuda.is_available()}")
print(f"Device name : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only - check Runtime > Change runtime type'}")

GPU available : True
Device name : Tesla T4


In [ ]:
!pip install ultralytics -q

#### ****2.5 Setup Processed Directory****


In [ ]:
!pip install imagehash -q

import os, shutil
import sys
sys.path.insert(0,os.path.join(PROJECT_DIR, 'src'))
from bonefracture.data_processing import rebuild_dataset_verified, convert_split_to_bbox

DEST = '/content/data/raw'
PROCESSED_DEST = '/content/data/processed'
os.makedirs(DEST, exist_ok=True)
os.makedirs(PROCESSED_DEST, exist_ok=True)

# Regenerate raw data (import your rebuild function or redefine it here)
success = rebuild_dataset_verified(DEST)
print(f"Raw data ready: {success}")

# Regenerate the bbox-converted version
for split in ['train', 'valid', 'test']:
    n = convert_split_to_bbox(DEST, PROCESSED_DEST, split)
    print(f"{split}: converted {n} label files")

shutil.copy2(os.path.join(DEST, 'data.yaml'), os.path.join(PROCESSED_DEST, 'data.yaml'))
print("data.yaml copied")

####**2.6 Train yolov8n with in built training functions**

In [ ]:
from ultralytics import YOLO
PROCESSED_DEST = '/content/data/processed'
model = YOLO('yolov8n.pt')

#Using YOLOs inbuilt training functionality
results = model.train(
    data=os.path.join(PROCESSED_DEST, 'data.yaml'),
    epochs=30,
    imgsz=640,
    batch=16,
    project=os.path.join(PROJECT_DIR, 'models'),
    name='baseline_yolov8n_v2',
    patience=10,
    weight_decay=0.001,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.2,
    degrees=5, translate=0.1, scale=0.2,
    shear=0.0, flipud=0.0, fliplr=0.5, mosaic=0.0,
)

### 2.7 **Load best model**

In [ ]:
from ultralytics import YOLO

best_model = YOLO(os.path.join(PROJECT_DIR, 'models', 'baseline_yolov8n-4', 'weights','best.pt'))

metrics = best_model.val(
    data = os.path.join(PROCESSED_DEST, 'data.yaml'),
    split ='test',
)

print(f"mAP50 : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map: .3f}")

#### **2.8 Evaluate on test set with in-built test function**


In [ ]:
import yaml
yaml_path = os.path.join(PROCESSED_DEST, 'data.yaml')
with open(yaml_path) as f:
  config = yaml.safe_load(f)
class_names = config['names']

metrics = model.val(
    data = os.path.join(PROCESSED_DEST, 'data.yaml'),
                                        split = 'test',)
print(f"\nmAP50 : {metrics.box.map50: .3f}")
print(f"mAP50-95 : {metrics.box.map: .3f}")
print(f"\nPer-class mAP50: ")

for i, class_name in enumerate(class_names):
  print(f" {class_name} : {metrics.box.ap50[i]: .3f}")

#### **2.9 Check on inference**


In [ ]:
from tqdm import tqdm
import random
from PIL import Image
import matplotlib.pyplot as plt

best_model = YOLO(os.path.join(PROJECT_DIR, 'models', 'baseline_yolov8n-4', 'weights', 'best.pt'))

test_images_dir = os.path.join(PROCESSED_DEST, 'test', 'images')
test_files = os.listdir(test_images_dir)
sample_files = random.sample(test_files, min(6, len(test_files)))

fig, axes = plt.subplots(2, 3, figsize=(18,10))
axes = axes.flatten()

for ax, filename in tqdm(zip(axes, sample_files), total=len(sample_files), desc = "Running inference"):
  img_path = os.path.join(test_images_dir, filename)
  result = best_model.predict(img_path, verbose=False)[0]

  #Ultralytics own .plot() draws predicted boxes + labels + confidence
  annotated = result.plot()
  ax.imshow(annotated)
  ax.set_title(filename, fontsize = 8)
  ax.axis('off')

plt.suptitle("Sample test predictions (baseline yolov8n)", fontsize =14)
plt.tight_layout()
plt.show()

In [ ]:
print(model.trainer.save_dir)